## University of Surrey
### MSc Data Science Dissertation - Emotion Aware Music Recommendation System with Explainable AI
###Model training and Benchmark evaluation

Name: Minal Honali Raghunandan


Student ID: 6908107


Notebook3 -
source 117 tracks via custom browser plugin, extract its acoustic features using openSMILE configuration. The resulting feature set to be aligned to the 302 column format expected by the tuned XGBoost model from model evaluation notebook. this is for real world validation, which is later used in notebook 4

In [1]:
!pip install -q yt-dlp opensmile
!apt-get install -y ffmpeg -q

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.


In [2]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = "/content/drive/MyDrive/Dissertation/data/plugin"
os.makedirs(DRIVE_PATH, exist_ok=True)

df_plugin = pd.read_csv('/content/plugin_data_final.csv')
df_plugin.to_csv(os.path.join(DRIVE_PATH, 'plugin_data_final.csv'), index=False)

print(f"Saved {len(df_plugin)} tracks to {DRIVE_PATH}")
print(df_plugin['Quadrant'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved 117 tracks to /content/drive/MyDrive/Dissertation/data/plugin
Quadrant
excited_elated    38
sad_depressed     28
angry_stressed    22
calm_relaxed      16
content_serene     8
bored_tired        5
Name: count, dtype: int64


In [3]:
import yt_dlp

AUDIO_DIR = "/content/plugin_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

def download_tracks(track, artist, out_dir=AUDIO_DIR):
  safe_name = f"{artist}_{track}".replace(" ", "_").replace("/", "_")[:80]
  out_path = os.path.join(out_dir, safe_name)
  wav_path = f"{out_path}.wav"

  if os.path.exists(wav_path):
    return wav_path

  query = f"{artist} {track} official audio"
  ydl_opts = {
      "format": "bestaudio/best",
      "outtmpl": f"{out_path}.%(ext)s",
      "postprocessors": [{
          "key": "FFmpegExtractAudio",
          "preferredcodec": "wav",
          "preferredquality": "192",
        }],
      "default_search": "ytsearch1",
      "quiet": True,
      "no_warnings": True,
      "noplaylist": True,
  }
  try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
      ydl.download([query])
    return wav_path
  except Exception as e:
    print(f"Error downloading {query}: {e}")
    return None

audio_path = []
for i, row in df_plugin.iterrows():
  audio_path.append(download_tracks(row['Track'], row['Artist']))
  if (i + 1) % 10 == 0:
    print(f"Downloaded {i+1}/{len(df_plugin)} tracks")
df_plugin['Audio Path'] = audio_path

success = df_plugin["Audio Path"].notna().sum()
print(f"\nDownloaded {success}/{len(df_plugin)} tracks successfully")

df_plugin.to_csv(os.path.join(DRIVE_PATH, 'plugin_data_final.csv'), index=False)
print("saved with audio path column")

Downloaded 10/117 tracks
Downloaded 20/117 tracks
Downloaded 30/117 tracks
Downloaded 40/117 tracks
Downloaded 50/117 tracks
Downloaded 60/117 tracks
Downloaded 70/117 tracks
Downloaded 80/117 tracks
Downloaded 90/117 tracks
Downloaded 100/117 tracks
Downloaded 110/117 tracks

Downloaded 117/117 tracks successfully
saved with audio path column


In [4]:
print(f"Downloaded: {success}/{len(df_plugin)}")
print(f"Failed tracks: {len(df_plugin[df_plugin['Audio Path'].isna()])}")

Downloaded: 117/117
Failed tracks: 0


In [5]:
import opensmile
pack = os.path.dirname(opensmile.__file__)
config = os.path.join(pack,"core/config/is09-13/IS13_ComParE.conf")

smile = opensmile.Smile(
    feature_set=config,
    feature_level="lld",
)

print(f"LLD feature count: {len(smile.feature_names)}")
print(smile.feature_names[:10])

LLD feature count: 65
['F0final_sma', 'voicingFinalUnclipped_sma', 'jitterLocal_sma', 'jitterDDP_sma', 'shimmerLocal_sma', 'logHNR_sma', 'audspec_lengthL1norm_sma', 'audspecRasta_lengthL1norm_sma', 'pcm_RMSenergy_sma', 'pcm_zcr_sma']


In [6]:
DATA_PATH = "/content/drive/MyDrive/Dissertation/data/plugin"
df_plugin = pd.read_csv(os.path.join(DATA_PATH, 'plugin_data_final.csv'))

agg_features = []
failed_extractions = []

for i, row in df_plugin.iterrows():
  audio_path = row["Audio Path"]
  if pd.isna(audio_path) or not os.path.exists(audio_path):
    failed_extractions.append(row["Track"])
    continue

  try:
    lld_df = smile.process_file(audio_path)

    mean = lld_df.mean()
    std = lld_df.std()
    mean.index = [f"{col}_mean" for col in mean.index]
    std.index = [f"{col}_std" for col in std.index]

    combined = pd.concat([mean, std])
    combined["song_id"] = row["song_id"]
    agg_features.append(combined)

    if (i + 1) % 10 == 0:
      print(f"Processed {i+1}/{len(df_plugin)}")

  except Exception as e:
    print(f"Error extracting features from {audio_path}: {e}")
    failed_extractions.append(row["Track"])

df_features = pd.DataFrame(agg_features)
print(f"\nExtracted: {len(df_features)}/{len(df_plugin)}")
print(f"Feature cols: {df_features.shape[1] - 1}")  # -1 for song_id

if failed_extractions:
  print(f"\nFailed tracks: {failed_extractions}")

Processed 10/117
Processed 20/117
Processed 30/117
Processed 40/117
Processed 50/117
Processed 60/117
Processed 70/117
Processed 80/117
Processed 90/117
Processed 100/117
Processed 110/117

Extracted: 117/117
Feature cols: 130


In [9]:
import joblib
MODEL_DIR = "/content/drive/MyDrive/Dissertation/models"
trained_cols = joblib.load(os.path.join(MODEL_DIR,"feature_columns.pkl"))

plugin_cols = set(df_features.columns) - {"song_id"}
trained_cols_set = set(trained_cols)

overlap = plugin_cols & trained_cols_set
missing = trained_cols_set - plugin_cols

print(f"Trained model expects: {len(trained_cols_set)}")
print(f"Plugin data has: {len(plugin_cols)} columns")
print(f"Overlap: {len(overlap)} columns")
print(f"Missing: {len(missing)}columns ")

print("Sample of overlapping columns:")
print(list(overlap)[:10])

Trained model expects: 302
Plugin data has: 130 columns
Overlap: 0 columns
Missing: 302columns 
Sample of overlapping columns:
[]


In [8]:
import os
MODEL_DIR = "/content/drive/MyDrive/Dissertation/models"
if os.path.exists(MODEL_DIR):
    print(os.listdir(MODEL_DIR))
else:
    print("Folder doesn't exist at all")

['valence_model.pkl', 'arousal_model.pkl', 'feature_columns.pkl']


In [11]:
print("PLUGIN feature names (sample):")
print(sorted(plugin_cols)[:10])
print()
print("TRAINED MODEL feature names (sample):")
print(sorted(trained_cols_set)[:10])

PLUGIN feature names (sample):
['F0final_sma_mean', 'F0final_sma_std', 'audSpec_Rfilt_sma[0]_mean', 'audSpec_Rfilt_sma[0]_std', 'audSpec_Rfilt_sma[10]_mean', 'audSpec_Rfilt_sma[10]_std', 'audSpec_Rfilt_sma[11]_mean', 'audSpec_Rfilt_sma[11]_std', 'audSpec_Rfilt_sma[12]_mean', 'audSpec_Rfilt_sma[12]_std']

TRAINED MODEL feature names (sample):
['F0final_sma_amean_mean', 'F0final_sma_amean_std', 'F0final_sma_de_amean_mean', 'F0final_sma_de_stddev_mean', 'F0final_sma_de_stddev_std', 'F0final_sma_stddev_mean', 'F0final_sma_stddev_std', 'audSpec_Rfilt_sma_0__stddev_mean', 'audSpec_Rfilt_sma_0__stddev_std', 'audSpec_Rfilt_sma_10__amean_mean']


I noticed that DEAM names embed extra functional layer like _amean, _stddev etc. hence there is 0 overlap.

I will now normalise the plugin names so that it matches with DEAM feature columns

In [17]:
import re
def normalise_plugin(name):
  name = re.sub(r'\[(\d+)]', r'_\1_', name)
  return name

plugin_normalised = {normalise_plugin(c): c for c in plugin_cols}
map_cols = {}
for trained_col in trained_cols_set:
  base = trained_col.replace('_amean_mean', '').replace('_amean_std', '') \
  .replace('_stddev_mean', '').replace('_stddev_std', '')
  if '_amean_mean' in trained_col:
    candidate = f"{base}_mean"
  elif '_stddev_mean' in trained_col:
    candidate = f"{base}_std"
  else:
    continue

  if candidate in plugin_normalised:
    map_cols[trained_col] = plugin_normalised[candidate]

print(f"successfully mapped: {len(map_cols)} / {len(trained_cols_set)} columns")
print(f"fill median: {len(trained_cols_set) - len(map_cols)} columns")

successfully mapped: 54 / 302 columns
fill median: 248 columns


In [15]:
print(type(plugin_normalised))
print(type(map_cols))

<class 'list'>
<class 'dict'>


In [22]:
import re
from sklearn.preprocessing import StandardScaler
import joblib

scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
# Recover raw training values — scaler was fit on ORIGINAL bracket-style column names
X_train_scaled_ref = pd.read_csv(os.path.join(DATA_PATH, "X_train.csv"))
raw_cols = X_train_scaled_ref.columns.tolist()  # bracket-style, scaler's expected order

X_train_raw = pd.DataFrame(scaler.inverse_transform(X_train_scaled_ref), columns=raw_cols)
train_median_raw = X_train_raw.median()

# Bridge: sanitized name (used by trained_cols/XGBoost) -> original bracket name (used by scaler)
regex_pattern = re.compile(r"\[|\]|<")
sanitized_to_raw = {regex_pattern.sub("_", c): c for c in raw_cols}

# Build aligned matrix using RAW bracket names, since that's what the scaler expects
aligned_rows = []
for _, row in df_features.iterrows():
    aligned = {}
    for trained_col in trained_cols:
        raw_col = sanitized_to_raw.get(trained_col, trained_col)
        if trained_col in map_cols:
            aligned[raw_col] = row[map_cols[trained_col]]
        else:
            aligned[raw_col] = train_median_raw[raw_col]
    aligned_rows.append(aligned)

X_plugin_aligned_raw = pd.DataFrame(aligned_rows, columns=raw_cols)
X_plugin_scaled_raw = pd.DataFrame(scaler.transform(X_plugin_aligned_raw), columns=raw_cols)

# Rename to sanitized names to match what the trained XGBoost models expect
X_plugin_scaled = X_plugin_scaled_raw.rename(columns=lambda c: regex_pattern.sub("_", c))
X_plugin_scaled = X_plugin_scaled[trained_cols]  # enforce exact training column order

print(f"X_plugin_scaled: {X_plugin_scaled.shape}")

X_plugin_scaled: (117, 302)


In [25]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
MODEL_DIR = "/content/drive/MyDrive/Dissertation/models"
DATA_PATH = "/content/drive/MyDrive/Dissertation/data/processed"

valence_model = joblib.load(os.path.join(MODEL_DIR, "valence_model.pkl"))
arousal_model = joblib.load(os.path.join(MODEL_DIR, "arousal_model.pkl"))
trained_cols = joblib.load(os.path.join(MODEL_DIR, "feature_columns.pkl"))
scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
y_pred_valence = valence_model.predict(X_plugin_scaled)
y_pred_arousal = arousal_model.predict(X_plugin_scaled)

y_true_valence = df_plugin["Target_Valence"].values
y_true_arousal = df_plugin["Target_Arousal"].values

def eval_plugin(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name} — R²: {round(r2,4)} | RMSE: {round(rmse,4)} | MAE: {round(mae,4)}")
    return {"Target": name, "R2": round(r2,4), "RMSE": round(rmse,4), "MAE": round(mae,4)}

results_plugin = []
results_plugin.append(eval_plugin(y_true_valence, y_pred_valence, "Valence"))
results_plugin.append(eval_plugin(y_true_arousal, y_pred_arousal, "Arousal"))

df_plugin_results = pd.DataFrame(results_plugin)

Valence — R²: 0.0123 | RMSE: 0.7412 | MAE: 0.7378
Arousal — R²: 0.0234 | RMSE: 0.725 | MAE: 0.7184


In [26]:
def to_quadrant(valence, arousal):
    if valence >= 0 and arousal >= 0:
        return "excited_elated"
    elif valence < 0 and arousal >= 0:
        return "angry_stressed"
    elif valence < 0 and arousal < 0:
        return "sad_depressed"
    else:
        return "calm_relaxed"  # covers calm_relaxed / content_serene / bored_tired

pred_quadrant = [to_quadrant(v, a) for v, a in zip(y_pred_valence, y_pred_arousal)]
true_quadrant_simplified = df_plugin["Quadrant"].replace({
    "content_serene": "calm_relaxed",
    "bored_tired": "calm_relaxed"
})

accuracy = (pd.Series(pred_quadrant) == true_quadrant_simplified.values).mean()
print(f"Quadrant-direction accuracy: {round(accuracy*100, 1)}%")

from collections import Counter
print("\nPredicted quadrant distribution:", Counter(pred_quadrant))

Quadrant-direction accuracy: 34.2%

Predicted quadrant distribution: Counter({'excited_elated': 104, 'angry_stressed': 7, 'sad_depressed': 5, 'calm_relaxed': 1})


A real-world validation step was attempted using a custom browser plugin that logged 117 self-reported mood labels during actual music listening sessions across Spotify and YouTube. However, reproducing DEAM's original acoustic feature extraction pipeline proved infeasible: DEAM's provided features rely on a proprietary windowing and functional-computation scheme (IS13_ComParE_lld-func.conf) that could not be exactly replicated using publicly available openSMILE tooling, resulting in only 54 of 302 trained features (18%) being reproducible from freshly-extracted audio.